### XGB Baseline

This notebook: builds the canonical May2024 stop-level data, split plan, and transformed stop-level dataset; trains the XGBoost benchmark model for SMT comparison; and evaluates the predictions using the shared inverse-transform evaluation path.

In [ ]:
# %reset -f
%load_ext autoreload
%autoreload complete --log


In [ ]:
from smtgraphformer import *
from smtgraphformer.benchmarks.xgb import *
from smtgraphformer.modelAdapters import createCanonicalBuilds

setDisplayOptions()
sr = setReproducibility(17711)


In [ ]:
root = Path("../data")
fp = root.joinpath("atbData-May2024-stopLevel-[fPM.eST.eLU.eDW].pkl")
assert fp.exists(), "!!!"


### Shared Artefacts

In [ ]:
builder = createCanonicalBuilds(fp)
# ---
dataBCS = builder.canonicalStops
ds_splits = builder.splitPlan
bundle = builder.transformBundle
dataFTB = builder.transformedStops


In [ ]:
xgbData = tfmStopLevelXGB(dataFTB, bundle)
printFeatureCardinality(xgbData)


### Training and Evaluation

In [ ]:
cfgXGB = XGBConfig(
    n_estimators=2048,
    patience=205,
    model_dir="../models/xgbM24",
)

with pipeCellOutput(f"{cfgXGB.model_dir}/training.log"):
    model, metrics, comparisons = runBaselineXGB(dataFTB, bundle, cfgXGB, raw_evaluation=True)

print(metrics.tail(4))


In [ ]:
model.save(verbose=True)
df_csver(metrics, "../models/benchmarks/metricsXGB")

# for s, comp in comparisons.items():
#     df_csver(comp, f"{cfgXGB.model_dir}/comparison-{s}")


### end